# UTS Data Acquisition & Management — Direktori Akomodasi Ubud

**Nama:** Ni Putu Riska Pramesti Cahya Dewi  
**NIM:** 2501010315  
**Program Studi:** Informatika — INSTIKI

Notebook ini menggunakan dataset hasil ekstraksi dari publikasi *Direktori Hotel Kabupaten Gianyar 2017* (data tahun 2015). Tidak ada data hotel atau desa buatan yang digunakan. Data cuaca diambil dari Open-Meteo Historical Weather API untuk 1–31 Januari 2025. Karena periode kedua sumber berbeda, cuaca hanya menjadi konteks wilayah, bukan kondisi cuaca pada tahun 2015.

**Penting:** Jalankan seluruh notebook dari awal sampai akhir. Seluruh output dibuat ulang dari file sumber dan respons API aktual. Jika sumber tidak tersedia atau API gagal, notebook akan berhenti dengan pesan yang jelas; notebook tidak membuat data pengganti.


## 1. Persiapan folder dan pustaka

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, time, subprocess
import requests
import pandas as pd
from IPython.display import display

# Google Colab tidak otomatis mengunduh folder data saat notebook dibuka dari GitHub.
# Karena itu, clone repository terlebih dahulu jika file sumber belum tersedia.
REPO_URL = "https://github.com/niputuriskapramesticahyadewi-maker/UTS_DAM_UBUD_RISKA_2501010315.git"
REPO_DIR = Path("/content/UTS_DAM_UBUD_RISKA_2501010315")
NAMA_FOLDER = "UTS_DAM_UBUD_RISKA_2501010315_FINAL"

def csv_tersedia(folder):
    return (folder / "data" / "interim" / "akomodasi_ubud_bps_2015_ekstraksi.csv").is_file()

# Coba folder kerja yang mungkin sudah ada.
kandidat = [
    Path.cwd() / NAMA_FOLDER,
    Path.cwd(),
    REPO_DIR / NAMA_FOLDER,
    Path("/content") / NAMA_FOLDER,
]
BASE_DIR = next((p for p in kandidat if csv_tersedia(p)), None)

# Jika belum ada, unduh repository GitHub beserta folder data.
if BASE_DIR is None:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    BASE_DIR = REPO_DIR / NAMA_FOLDER

if not csv_tersedia(BASE_DIR):
    raise FileNotFoundError(
        "CSV sumber masih belum ditemukan. Pastikan repository GitHub berisi "
        "data/interim/akomodasi_ubud_bps_2015_ekstraksi.csv."
    )

RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
META_DIR = BASE_DIR / "metadata"
DOCS_DIR = BASE_DIR / "docs"
for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, META_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_PATH = INTERIM_DIR / "akomodasi_ubud_bps_2015_ekstraksi.csv"
PDF_PATH = RAW_DIR / "BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf"
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"CSV sumber tidak ditemukan: {DATASET_PATH}")
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF sumber tidak ditemukan: {PDF_PATH}")

def get_checked(url, params=None, timeout=30):
    response = requests.get(
        url, params=params, timeout=timeout,
        headers={"User-Agent": "UTS-DAM-Student-Project/1.0"}
    )
    response.raise_for_status()
    return response

print("Folder proyek:", BASE_DIR)
print("CSV sumber tersedia:", DATASET_PATH.exists())
print("PDF BPS tersedia:", PDF_PATH.exists())


## 2. Membaca dan memvalidasi dataset BPS

In [ ]:
df_akomodasi = pd.read_csv(DATASET_PATH, encoding="utf-8-sig")
kolom_wajib = ["no_sumber", "desa", "nama_akomodasi", "alamat", "klasifikasi", "kecamatan", "tahun_data"]
kolom_hilang = sorted(set(kolom_wajib) - set(df_akomodasi.columns))
if kolom_hilang:
    raise ValueError(f"Kolom wajib tidak ditemukan: {kolom_hilang}")
if len(df_akomodasi) < 100:
    raise ValueError(f"Dataset hanya memiliki {len(df_akomodasi)} baris; minimal 100.")
if df_akomodasi["nama_akomodasi"].isna().any():
    raise ValueError("Ada nama akomodasi kosong.")
if df_akomodasi.duplicated().any():
    raise ValueError("Ditemukan baris duplikat pada dataset sumber.")
if df_akomodasi["nama_akomodasi"].astype(str).str.contains(r"Hotel Ubud Indah|Desa_Ubud_", case=False, regex=True).any():
    raise ValueError("Terindikasi data dummy. Hentikan analisis dan periksa sumber CSV.")

print("Jumlah baris:", len(df_akomodasi))
print("Jumlah kolom:", len(df_akomodasi.columns))
print("Tahun data:", sorted(df_akomodasi["tahun_data"].dropna().unique().tolist()))
print("Desa/kelurahan:", df_akomodasi["desa"].nunique())
print("Duplikasi:", int(df_akomodasi.duplicated().sum()))
display(df_akomodasi.head())
display(df_akomodasi["klasifikasi"].value_counts().to_frame("jumlah"))


## 3. Etika data dan checklist

In [ ]:
CHECKLIST_CSV = META_DIR / "data_ethics_checklist.csv"
if CHECKLIST_CSV.exists():
    df_checklist = pd.read_csv(CHECKLIST_CSV, encoding="utf-8-sig")
    display(df_checklist)
else:
    print("Checklist belum tersedia; buat dokumentasi checklist secara terpisah sebelum pengumpulan.")

print("Kolom yang berpotensi menjadi pengenal:", ["nama_akomodasi", "alamat", "desa"])
print("Kolom nomor telepon tersedia:", any("telepon" in str(c).lower() or "phone" in str(c).lower() for c in df_akomodasi.columns))
print("Catatan: data usaha/alamat bukan otomatis data pribadi; tetap batasi penggunaan pada tujuan akademik dan ikuti ketentuan sumber.");


## 4. Akuisisi data cuaca melalui API Open-Meteo

In [ ]:
API_URL = "https://archive-api.open-meteo.com/v1/archive"
parameter_api = {
    "latitude": -8.5069,
    "longitude": 115.2625,
    "start_date": "2025-01-01",
    "end_date": "2025-01-31",
    "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
    "timezone": "Asia/Makassar"
}
api_response = get_checked(API_URL, params=parameter_api)
raw_api = api_response.json()
if "daily" not in raw_api or "time" not in raw_api["daily"]:
    raise ValueError("Respons Open-Meteo tidak memiliki struktur daily/time yang diharapkan.")

daily = raw_api["daily"]
df_cuaca = pd.DataFrame({
    "tanggal": daily["time"],
    "suhu_rata2_c": daily["temperature_2m_mean"],
    "curah_hujan_mm": daily["precipitation_sum"],
    "kecepatan_angin_maks_kmh": daily["wind_speed_10m_max"]
})
if len(df_cuaca) < 30 or df_cuaca["tanggal"].duplicated().any():
    raise ValueError("Data cuaca tidak memenuhi minimal 30 tanggal unik.")

API_RAW_PATH = RAW_DIR / "open_meteo_ubud_2025-01.json"
API_RAW_PATH.write_text(json.dumps(raw_api, ensure_ascii=False, indent=2), encoding="utf-8")
CUACA_CSV_PATH = INTERIM_DIR / "cuaca_harian_ubud_2025-01.csv"
df_cuaca.to_csv(CUACA_CSV_PATH, index=False, encoding="utf-8-sig")

print("HTTP status:", api_response.status_code)
print("Jumlah observasi harian:", len(df_cuaca))
print("Periode:", df_cuaca["tanggal"].min(), "s.d.", df_cuaca["tanggal"].max())
display(df_cuaca.head())


## 5. Pemeriksaan robots.txt dan scraping etis

In [ ]:
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SCRAPE_BASE = "https://bali.bps.go.id"
ROBOTS_URL = urljoin(SCRAPE_BASE, "/robots.txt")
SCRAPE_URL = "https://bali.bps.go.id/"
robots_status = None
status_scraping = "BELUM DIPERIKSA"
robots_path = RAW_DIR / "robots_bali_bps.txt"

try:
    robots_response = requests.get(ROBOTS_URL, timeout=20, headers={"User-Agent":"UTS-DAM-Student-Project/1.0"})
    robots_status = robots_response.status_code
    if robots_response.ok:
        robots_path.write_text(robots_response.text, encoding="utf-8")
        parser = RobotFileParser()
        parser.parse(robots_response.text.splitlines())
        boleh = parser.can_fetch("UTS-DAM-Student-Project/1.0", SCRAPE_URL)
        status_scraping = "DIIZINKAN" if boleh else "TIDAK DIIZINKAN"
    else:
        status_scraping = f"TIDAK DILAKUKAN: robots.txt HTTP {robots_status}"
except requests.RequestException as exc:
    status_scraping = f"TIDAK DILAKUKAN: pemeriksaan robots.txt gagal ({type(exc).__name__}: {exc})"

bukti = (
    f"URL robots.txt: {ROBOTS_URL}\n"
    f"HTTP status: {robots_status}\n"
    f"Status: {status_scraping}\n"
    "Scraping tidak dilakukan jika robots.txt tidak dapat diperiksa atau akses tidak diizinkan.\n"
)
(DOCS_DIR / "bukti_scraping.txt").write_text(bukti, encoding="utf-8")
print(bukti)


## 6. Penggabungan data dan validasi

In [ ]:
# Ringkas cuaca harian menjadi satu baris untuk Kecamatan Ubud.
df_cuaca_ringkas = pd.DataFrame([{
    "kecamatan": "Ubud",
    "periode_cuaca": "2025-01",
    "suhu_rata2_bulanan_c": df_cuaca["suhu_rata2_c"].mean(),
    "total_curah_hujan_mm": df_cuaca["curah_hujan_mm"].sum(),
    "angin_maks_kmh": df_cuaca["kecepatan_angin_maks_kmh"].max()
}])
df_hasil = df_akomodasi.merge(df_cuaca_ringkas, on="kecamatan", how="left", validate="many_to_one", indicator=True)
jumlah_cocok = int((df_hasil["_merge"] == "both").sum())
jumlah_tidak_cocok = int((df_hasil["_merge"] != "both").sum())
df_hasil = df_hasil.drop(columns=["_merge"])
if len(df_hasil) != len(df_akomodasi) or jumlah_tidak_cocok:
    raise ValueError("Validasi merge gagal.")
if df_hasil.duplicated().any():
    raise ValueError("Ditemukan duplikasi pada hasil merge.")

OUTPUT_MERGE = PROCESSED_DIR / "akomodasi_cuaca_ubud.csv"
df_hasil.to_csv(OUTPUT_MERGE, index=False, encoding="utf-8-sig")
print("Baris sumber:", len(df_akomodasi))
print("Baris hasil:", len(df_hasil))
print("Baris cocok:", jumlah_cocok)
print("Baris tidak cocok:", jumlah_tidak_cocok)
print("Jumlah kolom hasil:", len(df_hasil.columns))
display(df_hasil.head())


## 7. Provenance dan hash MD5

In [ ]:
def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

file_sumber = [
    (PDF_PATH, "BPS Kabupaten Gianyar", "PDF", "https://gianyarkab.bps.go.id/", "Publikasi Direktori Hotel Kabupaten Gianyar 2017; data direktori tahun 2015."),
    (DATASET_PATH, "Ekstraksi publikasi BPS", "CSV", "Dokumentasi ekstraksi dari PDF BPS", "Dataset akomodasi Ubud hasil ekstraksi; periksa kesesuaian setiap baris dengan PDF."),
    (API_RAW_PATH, "Open-Meteo Historical Weather API", "JSON", API_URL, "Respons API mentah untuk 1–31 Januari 2025."),
    (CUACA_CSV_PATH, "Open-Meteo Historical Weather API", "CSV", API_URL, "Data cuaca harian yang diturunkan dari JSON mentah."),
    (OUTPUT_MERGE, "Hasil pengolahan notebook", "CSV", "Proses merge lokal", "Penggabungan many-to-one akomodasi dan ringkasan cuaca Ubud.")
]
provenance = []
for path, source, kind, url, note in file_sumber:
    if path.exists():
        provenance.append({
            "sumber": source, "jenis": kind, "url": url,
            "tanggal_pembuatan_atau_akses": datetime.now(timezone.utc).isoformat(),
            "catatan": note, "file": path.relative_to(BASE_DIR).as_posix(),
            "ukuran_byte": path.stat().st_size, "md5": md5_file(path)
        })
    else:
        print("Tidak dicatat karena file belum ada:", path)

(META_DIR / "provenance.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(provenance).to_csv(META_DIR / "provenance.csv", index=False, encoding="utf-8-sig")
print("Jumlah file tercatat:", len(provenance))
display(pd.DataFrame(provenance))


## 8. Kesimpulan

In [ ]:
print("RINGKASAN UTS DAM")
print("=" * 45)
print(f"Dataset akomodasi BPS: {len(df_akomodasi)} baris, {len(df_akomodasi.columns)} kolom")
print(f"Data cuaca Open-Meteo: {len(df_cuaca)} observasi harian")
print(f"Status pemeriksaan robots.txt/scraping: {status_scraping}")
print(f"Hasil penggabungan: {len(df_hasil)} baris, {len(df_hasil.columns)} kolom")
print(f"Baris cocok: {jumlah_cocok}; tidak cocok: {jumlah_tidak_cocok}")
print(f"Duplikasi hasil: {int(df_hasil.duplicated().sum())}")
print("Perhatian: data akomodasi tahun 2015 dan cuaca Januari 2025 berbeda periode.")
print("Periksa file output, provenance, dan checklist etika sebelum pengumpulan.")
